# Sage benchmark search — NDP workspace

This standalone notebook optionally downloads all five benchmark datasets, compares **edge_v1** and **edge_v2**, and returns up to **25 visual search results from each version**. Both original exported indexes are restored into embedded Milvus Lite vector databases; neither version is rebuilt with a different embedding model.

**edge_v1 models:** `gemma-3-4b-it` generates detailed captions of roughly 150 words through vLLM. `apple/DFN5B-CLIP-ViT-H-14-378` embeds each image into a 1024-dimensional vector. Captions remain text and contribute through BM25; there is no dense caption vector. Benchmark fusion is 75% image vector and 25% caption BM25.

**edge_v2 models:** Ollama `gemma4:e2b`, with thinking enabled, generates shorter captions of roughly 50 words so they fit DFN5B-CLIP's 77-token text limit. The same `apple/DFN5B-CLIP-ViT-H-14-378` model produces both the 1024-dimensional image vector and the 1024-dimensional caption vector. Benchmark fusion is 60% image vector, 25% caption vector, and 15% caption BM25.

The model ID is stored with each index so query vectors always use the matching encoder. Before the custom-query demo, the notebook generates fresh results from both local indexes and compares them with every bundled baseline/v10/v11/v12 per-query result. It does not reuse saved edge_v1 or edge_v2 benchmark scores.


## 1. Install this notebook's requirements

Run once after cloning the repository, then restart the kernel if Jupyter asks.

In [ ]:
from pathlib import Path

candidates = [
    Path.cwd(),
    Path.cwd() / "notebooks" / "ndp_workspace",
]
NOTEBOOK_DIR = next(path.resolve() for path in candidates if (path / "requirements.txt").is_file())
REQUIREMENTS = NOTEBOOK_DIR / "requirements.txt"
print(f"Notebook directory: {NOTEBOOK_DIR}")
%pip install -q -r {REQUIREMENTS}

## 2. Configuration

Copy `.env.example` to `.env` and put the Hugging Face token there. This cell asks whether benchmark data should be downloaded and where the two exported NPZ files are located. Both edge_v1 and edge_v2 are restore-only so the comparison always uses their original DFN5B-CLIP vectors.

The fixed reproducibility weights are edge_v1 `0.75/0/0.25` and edge_v2 `0.60/0.25/0.15` for image vector, caption vector, and BM25 respectively.


In [ ]:
import os
import sys
from pathlib import Path

from dotenv import load_dotenv

if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))
load_dotenv(NOTEBOOK_DIR / ".env")

from notebook_helpers import (
    DATASETS,
    TextImageEncoder,
    benchmark_table,
    download_benchmarks,
    ensure_milvus_vector_database,
    evaluate_benchmarks,
    load_result_images,
    load_portable_index,
    load_reference_results,
    overall_table,
    query_result_table,
    score_bar_charts,
    search_milvus_database,
    show_results,
)

def ask_choice(prompt: str, choices: dict):
    while True:
        answer = input(prompt).strip().lower()
        if answer in choices:
            return choices[answer]
        print(f"Choose one of: {', '.join(choices)}")

HF_TOKEN = os.getenv("HF_TOKEN") or None
WORKSPACE_DATA = NOTEBOOK_DIR / "data"
HF_HOME = WORKSPACE_DATA / "huggingface"
os.environ["HF_HOME"] = str(HF_HOME)

DATASET_ROOT = WORKSPACE_DATA / "benchmarking" / "datasets"
# Both versions are restored from their completed portable exports.
DEFAULT_EDGE_V1_BACKUP = WORKSPACE_DATA / "edge_v1_benchmarks.npz"
DEFAULT_EDGE_V2_BACKUP = WORKSPACE_DATA / "edge_v2_benchmarks.npz"
EDGE_V1_DATABASE_FILE = WORKSPACE_DATA / "vector_database" / "edge_v1_benchmarks.milvus.db"
EDGE_V2_DATABASE_FILE = WORKSPACE_DATA / "vector_database" / "edge_v2_benchmarks.milvus.db"
DEVICE = "auto"
BATCH_SIZE = 64
TOP_K = 25
EDGE_V1_WEIGHTS = dict(image_weight=0.75, caption_weight=0.0, bm25_weight=0.25)
EDGE_V2_WEIGHTS = dict(image_weight=0.60, caption_weight=0.25, bm25_weight=0.15)
GENERATED_BENCHMARK_ROOT = WORKSPACE_DATA / "generated_benchmarks"
REFERENCE_RESULT_ROOT = NOTEBOOK_DIR / "results" / "benchmarks"

DOWNLOAD_BENCHMARKS = ask_choice(
    "Download all benchmark datasets from Hugging Face? [yes/no]: ",
    {"yes": True, "y": True, "no": False, "n": False},
)
if DOWNLOAD_BENCHMARKS and not HF_TOKEN:
    raise RuntimeError("Set HF_TOKEN in ndp_workspace/.env before downloading.")
entered = input(f"edge_v1 exported NPZ path [{DEFAULT_EDGE_V1_BACKUP}]: ").strip()
EDGE_V1_BACKUP = Path(entered).expanduser() if entered else DEFAULT_EDGE_V1_BACKUP
if not EDGE_V1_BACKUP.is_absolute():
    EDGE_V1_BACKUP = (NOTEBOOK_DIR / EDGE_V1_BACKUP).resolve()
if not EDGE_V1_BACKUP.is_file():
    raise FileNotFoundError(f"edge_v1 export not found: {EDGE_V1_BACKUP}")
entered = input(f"edge_v2 exported NPZ path [{DEFAULT_EDGE_V2_BACKUP}]: ").strip()
EDGE_V2_BACKUP = Path(entered).expanduser() if entered else DEFAULT_EDGE_V2_BACKUP
if not EDGE_V2_BACKUP.is_absolute():
    EDGE_V2_BACKUP = (NOTEBOOK_DIR / EDGE_V2_BACKUP).resolve()
if not EDGE_V2_BACKUP.is_file():
    raise FileNotFoundError(f"edge_v2 export not found: {EDGE_V2_BACKUP}")
print({
    "download_benchmarks": DOWNLOAD_BENCHMARKS,
    "datasets": list(DATASETS),
    "edge_v1_npz": str(EDGE_V1_BACKUP),
    "edge_v2_npz": str(EDGE_V2_BACKUP),
    "edge_v1_milvus": str(EDGE_V1_DATABASE_FILE),
    "edge_v2_milvus": str(EDGE_V2_DATABASE_FILE),
    "top_k": TOP_K,
})

## 3. Download or reuse the benchmark Parquet datasets

The exact Hugging Face revisions are pinned in `notebook_helpers.py`. If the user answered **yes**, all five datasets are downloaded with `HF_TOKEN`. If the answer was **no**, the cell verifies that the standalone workspace already contains the Parquet files. Benchmark scoring reads only ID and relevance columns. Images are not extracted; after a custom search, only the returned images are read from their Parquet row groups.

In [ ]:
if DOWNLOAD_BENCHMARKS:
    download_benchmarks(DATASET_ROOT, token=HF_TOKEN)
else:
    missing = [
        name for name in DATASETS
        if not list((DATASET_ROOT / name / "data").glob("*.parquet"))
    ]
    if missing:
        raise FileNotFoundError(
            f"Benchmark data is absent for {missing}. Rerun configuration and answer yes."
        )
print("Benchmark Parquet files are ready; full image extraction is skipped.")

## 4. Restore vectors, then populate the local database

For **edge_v1**, the notebook always restores `edge_v1_benchmarks.npz`, containing the original DFN5B image vectors and Gemma 3 caption text. It never substitutes MobileCLIP or rebuilds the vectors.

For **edge_v2**, the notebook always restores `edge_v2_benchmarks.npz`, exported from the completed run. It contains caption text plus both DFN5B image and caption vectors; Gemma and DFN5B are not rerun.

Each version is copied into its own embedded Milvus Lite database file—no container or database server. Both collections store the caption text and a built-in BM25 sparse field. edge_v1 stores its image vector only; edge_v2 stores both image and caption vectors. Existing compatible database files are reused.


In [ ]:
edge_v1_portable = load_portable_index(EDGE_V1_BACKUP)
edge_v1_manifest = __import__("json").loads(
    (NOTEBOOK_DIR / "edge_v1_export_manifest.json").read_text()
)
if edge_v1_portable.source != "edge_v1" or edge_v1_portable.has_caption_vectors:
    raise ValueError("The edge_v1 export must contain image vectors only")
if edge_v1_portable.model_id != edge_v1_manifest["model_id"]:
    raise ValueError("The edge_v1 export uses an unexpected embedding model")
if len(edge_v1_portable.records) != edge_v1_manifest["record_count"]:
    raise ValueError("The edge_v1 export does not contain the complete corpus")

edge_v2_portable = load_portable_index(EDGE_V2_BACKUP)
edge_v2_manifest = __import__("json").loads(
    (NOTEBOOK_DIR / "edge_v2_export_manifest.json").read_text()
)
if edge_v2_portable.source != "edge_v2" or not edge_v2_portable.has_caption_vectors:
    raise ValueError("The edge_v2 export must contain image and caption vectors")
if edge_v2_portable.model_id != edge_v2_manifest["model_id"]:
    raise ValueError("The edge_v2 export uses an unexpected embedding model")
if len(edge_v2_portable.records) != edge_v2_manifest["record_count"]:
    raise ValueError("The edge_v2 export does not contain the complete corpus")
edge_v1_database = ensure_milvus_vector_database(edge_v1_portable, EDGE_V1_DATABASE_FILE)
edge_v2_database = ensure_milvus_vector_database(edge_v2_portable, EDGE_V2_DATABASE_FILE)
edge_v1_index = edge_v1_portable
edge_v2_index = edge_v2_portable

dataset_counts = {
    version: {
        name: sum(record.dataset == name for record in selected.records)
        for name in DATASETS
    }
    for version, selected in {"edge_v1": edge_v1_index, "edge_v2": edge_v2_index}.items()
}
print(dataset_counts)

## 5. Load the matching query encoder

Each Milvus database records its matching query encoder. The encoder is shared only when both indexes use the same model ID, preventing vectors from different models from being mixed.

In [ ]:
encoders = {}
for selected in (edge_v1_index, edge_v2_index):
    if selected.model_id not in encoders:
        encoders[selected.model_id] = TextImageEncoder(
            selected.model_id, device=DEVICE, token=HF_TOKEN
        )
edge_v1_encoder = encoders[edge_v1_index.model_id]
edge_v2_encoder = encoders[edge_v2_index.model_id]

## 6. Generate a fresh benchmark run

This creates new edge_v1 and edge_v2 reproducibility results from the restored vectors and public relevance labels. It reports MRR, Success@25, Diversity@25, the two-metric primary score, and the three-metric score. Every baseline/v10/v11/v12 per-query result is bundled under `results/benchmarks`; saved edge result scores are not reused. The custom-query section uses Milvus's native BM25; its analyzer can differ slightly from the historical benchmark's `rank-bm25` implementation.

In [ ]:
edge_v1_label = f"edge_v1 (generated; {edge_v1_index.model_id.rsplit('/', 1)[-1]})"
edge_v2_label = f"edge_v2 (generated; {edge_v2_index.model_id.rsplit('/', 1)[-1]})"
edge_v1_summaries, edge_v1_query_rows = evaluate_benchmarks(
    index=edge_v1_index,
    encoder=edge_v1_encoder,
    dataset_root=DATASET_ROOT,
    output_root=GENERATED_BENCHMARK_ROOT / "edge_v1",
    top_k=TOP_K,
    batch_size=BATCH_SIZE,
    system_version=edge_v1_label,
    fusion_mode="topk",
    **EDGE_V1_WEIGHTS,
)
edge_v2_summaries, edge_v2_query_rows = evaluate_benchmarks(
    index=edge_v2_index,
    encoder=edge_v2_encoder,
    dataset_root=DATASET_ROOT,
    output_root=GENERATED_BENCHMARK_ROOT / "edge_v2",
    top_k=TOP_K,
    batch_size=BATCH_SIZE,
    system_version=edge_v2_label,
    fusion_mode="full_corpus",
    **EDGE_V2_WEIGHTS,
)
reference_summaries, reference_query_rows = load_reference_results(REFERENCE_RESULT_ROOT)
comparison_rows = reference_summaries + edge_v1_summaries + edge_v2_summaries
all_query_rows = reference_query_rows + edge_v1_query_rows + edge_v2_query_rows

import pandas as pd
ALL_RESULTS_FILE = GENERATED_BENCHMARK_ROOT / "all_comparison_query_results.csv"
pd.DataFrame(all_query_rows).to_csv(ALL_RESULTS_FILE, index=False)
print(f"Saved all {len(all_query_rows):,} per-query rows to {ALL_RESULTS_FILE}")

### FireBench — summary metrics and every per-query result

In [ ]:
display(benchmark_table(comparison_rows, "Firebench").round(4))
score_bar_charts(comparison_rows, "Firebench")
query_result_table(all_query_rows, "Firebench")

### CloudBench — summary metrics and every per-query result

In [ ]:
display(benchmark_table(comparison_rows, "Cloudbench").round(4))
score_bar_charts(comparison_rows, "Cloudbench")
query_result_table(all_query_rows, "Cloudbench")

### INQUIRE — summary metrics and every per-query result

In [ ]:
display(benchmark_table(comparison_rows, "INQUIRE").round(4))
score_bar_charts(comparison_rows, "INQUIRE")
query_result_table(all_query_rows, "INQUIRE")

### CommonObjectsBench — summary metrics and every per-query result

In [ ]:
display(benchmark_table(comparison_rows, "Commonobjectsbench").round(4))
score_bar_charts(comparison_rows, "Commonobjectsbench")
query_result_table(all_query_rows, "Commonobjectsbench")

### SageBench — summary metrics and every per-query result

In [ ]:
display(benchmark_table(comparison_rows, "Sagebench").round(4))
score_bar_charts(comparison_rows, "Sagebench")
query_result_table(all_query_rows, "Sagebench")

### Overall — equal weight across all five benchmarks

The overall table includes MRR, Success@25, Diversity@25, the two-metric primary score, and the equal-weight three-metric score. `benchmark_count` makes incomplete systems visible. The paired charts show both composite scores side by side for every system.

In [ ]:
display(overall_table(comparison_rows).round(4))
score_bar_charts(comparison_rows);

## 7. Custom query — 25 visual results

Enter one text query and run it against both Milvus Lite databases. Milvus searches the dense vector fields and its built-in BM25 sparse field; edge_v2 additionally searches its caption-vector field. Only the returned images are loaded directly from Parquet, so the notebook never extracts the complete image corpus.

In [ ]:
QUERY = input("Enter your image-search query: ").strip()
if not QUERY:
    raise ValueError("Query cannot be empty")

edge_v1_results = search_milvus_database(
    database=edge_v1_database,
    encoder=edge_v1_encoder,
    query=QUERY,
    top_k=TOP_K,
    **EDGE_V1_WEIGHTS,
)
edge_v2_results = search_milvus_database(
    database=edge_v2_database,
    encoder=edge_v2_encoder,
    query=QUERY,
    top_k=TOP_K,
    **EDGE_V2_WEIGHTS,
)
result_images = load_result_images(edge_v1_results + edge_v2_results, DATASET_ROOT)
print("edge_v1 search results")
show_results(edge_v1_results, result_images)
print("edge_v2 search results")
show_results(edge_v2_results, result_images)
{"edge_v1": edge_v1_results[:3], "edge_v2": edge_v2_results[:3]}